# Lab 3: Cleaning Multi-Source Retail Sales Data

Domain: Retail Analytics | Platform: R / Google Colab

In [32]:
library(readxl)

df_retail <- read_excel("/content/Online Retail.xlsx")

write.csv(df_retail, "/content/Online Retail.csv", row.names = FALSE)

print("Online Retail.xlsx successfully converted to Online Retail.csv")

[1] "Online Retail.xlsx successfully converted to Online Retail.csv"


In [33]:
if (!require("dplyr")) install.packages("dplyr")
if (!require("jsonlite")) install.packages("jsonlite")
if (!require("readxl")) install.packages("readxl")
if (!require("RSQLite")) install.packages("RSQLite")
if (!require("writexl")) install.packages("writexl") # Added for writing Excel files
if (!require("openxlsx")) install.packages("openxlsx") # Added for alternative Excel reading

library(dplyr)
library(jsonlite)
library(readxl)
library(RSQLite)
library(writexl) # Added
library(openxlsx) # Added

# Load the main dataset from 'Online Retail.xlsx'
retail_data_raw <- read.xlsx("/content/Online Retail.xlsx")

# Extract and clean transactions data
transactions_clean <- retail_data_raw %>%
  filter(!is.na(CustomerID)) %>% # Filter out rows with NA CustomerID for transactions
  select(InvoiceNo, StockCode, Quantity, InvoiceDate, UnitPrice, CustomerID, Country)

# Extract and clean products data
products_clean <- retail_data_raw %>% # Use retail_data_raw as source
  select(StockCode, Description, UnitPrice) %>%
  distinct(StockCode, .keep_all = TRUE) %>% # Keep unique products based on StockCode
  filter(!is.na(StockCode), UnitPrice > 0) # Ensure StockCode is not NA and UnitPrice is positive

# Ensure StockCode in products_clean is character type for successful join
products_clean$StockCode <- as.character(products_clean$StockCode)

# Extract and clean customers data
customers_clean <- retail_data_raw %>% # Use retail_data_raw as source
  filter(!is.na(CustomerID)) %>% # Filter out rows with NA CustomerID
  select(CustomerID, Country) %>%
  distinct(CustomerID, .keep_all = TRUE)

# Connect to the existing SQLite database. Assuming 'retail_sales.sqlite' is supplementary data.
# If this is not needed, this block can be removed.
conn <- dbConnect(SQLite(), "retail_sales.sqlite")
sales_db <- dbGetQuery(conn, "SELECT * FROM sales")
dbDisconnect(conn)

# Check dimensions
dims_transactions <- dim(transactions_clean)
dims_customers <- dim(customers_clean)
dims_products <- dim(products_clean)
dims_sales_db <- dim(sales_db)

cat("Dimensions of transactions_clean:", dims_transactions[1], "rows,", dims_transactions[2], "columns\n")
cat("Dimensions of customers_clean:", dims_customers[1], "rows,", dims_customers[2], "columns\n")
cat("Dimensions of products_clean:", dims_products[1], "rows,", dims_products[2], "columns\n")
cat("Dimensions of sales_db (from SQLite):", dims_sales_db[1], "rows,", dims_sales_db[2], "columns\n")

Dimensions of transactions_clean: 406829 rows, 7 columns
Dimensions of customers_clean: 4372 rows, 2 columns
Dimensions of products_clean: 3855 rows, 3 columns
Dimensions of sales_db (from SQLite): 3 rows, 3 columns


In [34]:
if (!require("dplyr")) install.packages("dplyr")
if (!require("jsonlite")) install.packages("jsonlite")
if (!require("readxl")) install.packages("readxl")
if (!require("RSQLite")) install.packages("RSQLite")
if (!require("writexl")) install.packages("writexl")

library(dplyr)
library(jsonlite)
library(readxl)
library(RSQLite)
library(writexl)

# Load the main dataset from 'Online Retail.csv' (converted from .xlsx by Python)
retail_data_raw <- read.csv("/content/Online Retail.csv", header = TRUE, stringsAsFactors = FALSE)

# Ensure InvoiceDate is properly converted to datetime if it's treated as character after CSV read
# If it is still a character, convert it here. If pandas already did it correctly, this might not be strictly necessary.
# For now, we'll assume read.csv keeps it in a format that can be handled later or if needed, converted using as.POSIXct.
# retail_data_raw$InvoiceDate <- as.POSIXct(retail_data_raw$InvoiceDate, format="%Y-%m-%d %H:%M:%S") # Example format

# Extract and clean transactions data
transactions_clean <- retail_data_raw %>%
  filter(!is.na(CustomerID)) %>%
  select(InvoiceNo, StockCode, Quantity, InvoiceDate, UnitPrice, CustomerID, Country)

# Extract and clean products data
products_clean <- retail_data_raw %>%
  select(StockCode, Description, UnitPrice) %>%
  distinct(StockCode, .keep_all = TRUE) %>%
  filter(!is.na(StockCode), UnitPrice > 0)

# Ensure StockCode in products_clean is character type for successful join
products_clean$StockCode <- as.character(products_clean$StockCode)

# Extract and clean customers data
customers_clean <- retail_data_raw %>%
  filter(!is.na(CustomerID)) %>%
  select(CustomerID, Country) %>%
  distinct(CustomerID, .keep_all = TRUE)

# Connect to the existing SQLite database. Assuming 'retail_sales.sqlite' is supplementary data.
# If this is not needed, this block can be removed.
conn <- dbConnect(SQLite(), "retail_sales.sqlite")
sales_db <- dbGetQuery(conn, "SELECT * FROM sales")
dbDisconnect(conn)

# Check dimensions
dims_transactions <- dim(transactions_clean)
dims_customers <- dim(customers_clean)
dims_products <- dim(products_clean)
dims_sales_db <- dim(sales_db)

cat("Dimensions of transactions_clean:", dims_transactions[1], "rows,", dims_transactions[2], "columns\n")
cat("Dimensions of customers_clean:", dims_customers[1], "rows,", dims_customers[2], "columns\n")
cat("Dimensions of products_clean:", dims_products[1], "rows,", dims_products[2], "columns\n")
cat("Dimensions of sales_db (from SQLite):", dims_sales_db[1], "rows,", dims_sales_db[2], "columns\n")

Dimensions of transactions_clean: 406829 rows, 7 columns
Dimensions of customers_clean: 4372 rows, 2 columns
Dimensions of products_clean: 3855 rows, 3 columns
Dimensions of sales_db (from SQLite): 3 rows, 3 columns


## Task 2: Integrate the Multiple Data Sources & Clean

In [35]:
# Data cleaning
transactions_clean <- transactions_clean %>%
  filter(!is.na(CustomerID), !is.na(StockCode)) %>%
  distinct()

products_clean <- products_clean %>%
  distinct(StockCode, .keep_all = TRUE) %>%
  filter(UnitPrice > 0)

customers_clean <- customers_clean %>%
  distinct(CustomerID, .keep_all = TRUE)

# Integrate using joins
final_data <- transactions_clean %>%
  inner_join(products_clean, by = "StockCode") %>%
  inner_join(customers_clean, by = "CustomerID")

# Derived column
final_data <- final_data %>%
  mutate(Revenue = Quantity * UnitPrice.x)

# Identify unmatched values
unmatched_products <- transactions_clean %>% anti_join(products_clean, by="StockCode")
unmatched_customers <- transactions_clean %>% anti_join(customers_clean, by="CustomerID")

## Task 3: Perform Sales and Customer Analysis

In [37]:
# 1. Total sales revenue
total_revenue <- sum(final_data$Revenue)
cat("Total Revenue:", total_revenue, "\n")

# 2. Top 5 products based on revenue
top_products <- final_data %>%
  group_by(StockCode, Description) %>%
  summarise(ProductRevenue = sum(Revenue), .groups = 'drop') %>%
  arrange(desc(ProductRevenue)) %>%
  head(5)
print(top_products)

# 3. Top 5 customers based on purchase value
top_customers <- final_data %>%
  group_by(CustomerID) %>%
  summarise(TotalPurchase = sum(Revenue), .groups = 'drop') %>%
  arrange(desc(TotalPurchase)) %>%
  head(5)
print(top_customers)

# 4. Classify customers based on Purchase
customer_summary <- final_data %>%
  group_by(CustomerID, Country.x) %>% # Changed Country to Country.x
  summarise(TotalPurchase = sum(Revenue), .groups = 'drop') %>%
  mutate(
    CustomerCategory = case_when(
      TotalPurchase < 1000 ~ "Low Value",
      TotalPurchase >= 1000 & TotalPurchase < 5000 ~ "Medium Value",
      TotalPurchase >= 5000 & TotalPurchase < 10000 ~ "High Value",
      TotalPurchase >= 10000 ~ "Premium"
    )
  )
print(customer_summary)

# 5. Market performance (Revenue by Country)
market_performance <- final_data %>%
  group_by(Country.x) %>% # Changed Country to Country.x
  summarise(MarketRevenue = sum(Revenue), .groups = 'drop') %>%
  arrange(desc(MarketRevenue))

print(market_performance)

Total Revenue: 8163532 
# A tibble: 5 × 3
  StockCode Description                        ProductRevenue
  <chr>     <chr>                                       <dbl>
1 22423     REGENCY CAKESTAND 3 TIER                  132568.
2 85123A    WHITE HANGING HEART T-LIGHT HOLDER         93923.
3 85099B    JUMBO BAG RED RETROSPOT                    83057.
4 47566     PARTY BUNTING                              67628.
5 POST      POSTAGE                                    66710.
# A tibble: 5 × 2
  CustomerID TotalPurchase
       <int>         <dbl>
1      14646       276183.
2      18102       256259.
3      17450       187322.
4      14911       131127.
5      12415       122388.
# A tibble: 4,380 × 4
   CustomerID Country.x      TotalPurchase CustomerCategory
        <int> <chr>                  <dbl> <chr>           
 1      12346 United Kingdom            0  Low Value       
 2      12347 Iceland                4310  Medium Value    
 3      12348 Finland                1797. Medium Value

## Task 4: Store and Retrieve Data Using SQL & Final Insights

In [40]:
# Create SQLite db and export final dataset
con <- dbConnect(RSQLite::SQLite(), "retail_sales.sqlite")

# Rename columns with periods to avoid issues in SQLite
final_data_db <- final_data %>%
  rename(Country_x = Country.x,
         Country_y = Country.y)

dbWriteTable(con, "retail_sales", final_data_db, overwrite = TRUE)

# Query 1: Top 5 customers based on revenue
query1 <- "
  SELECT CustomerID, SUM(Revenue) as TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5
"
print(dbGetQuery(con, query1))

# Query 2: Total revenue by country
query2 <- "
  SELECT Country_x, SUM(Revenue) as TotalRevenue
  FROM retail_sales
  GROUP BY Country_x
  ORDER BY TotalRevenue DESC
"
print(head(dbGetQuery(con, query2), 5))

dbDisconnect(con)

  CustomerID TotalRevenue
1      14646     276183.3
2      18102     256259.5
3      17450     187322.2
4      14911     131126.7
5      12415     122388.1
       Country_x TotalRevenue
1 United Kingdom    6647761.3
2    Netherlands     281318.5
3           EIRE     248414.3
4        Germany     219049.7
5         France     193048.7
